In [ ]:
import os
import sys
import json
import logging
from oandapyV20 import API
from oandapyV20.endpoints.transactions import TransactionIDRange
from oandapyV20.endpoints.trades import OpenTrades

# Setup paths
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))



from datetime import datetime

from pyairtable import Api
from config.oanda_config import API_KEY, ACCOUNT_ID
from config.airtable_config import AIRTABLE_API_TOKEN, BASE_ID, TABLE_NAME

# Airtable setup
airtable_api = Api(AIRTABLE_API_TOKEN)
table = airtable_api.table(BASE_ID, TABLE_NAME)


# Setup logging
LOG_FILE = os.path.join(os.path.dirname(__file__), "../logs/sync.log")
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode='a'
)
logger = logging.getLogger(__name__)
table = get_airtable_table(AIRTABLE_KEY, AIRTABLE_BASE_ID, AIRTABLE_TABLE_NAME)

# Store take-profit and stop-loss price mappings for reuse by trade ID
take_profit_by_trade_id = {}
stop_loss_by_trade_id = {}

def sync_new_fills():
    logger.info("📄 Syncing new fills...")
    client = API(access_token=API_KEY, environment="practice")

    last_id = load_last_transaction_id()
    params = {"from": str(int(last_id) + 1), "to": "99999999"}
    r = TransactionsSinceID(accountID=ACCOUNT_ID, params=params)
    client.request(r)

    transactions = r.response.get("transactions", [])
    logger.info(f"🔁 {len(transactions)} transactions received.")

    latest_tx_id = last_id

    for tx in transactions:
        latest_tx_id = tx["id"]
        logger.info("📩 Transaction received:\n%s", json.dumps(tx, indent=2))

        tx_type = tx["type"]

        if tx_type == "TAKE_PROFIT_ORDER":
            take_profit_by_trade_id[tx["tradeID"]] = tx
        elif tx_type == "STOP_LOSS_ORDER":
            stop_loss_by_trade_id[tx["tradeID"]] = tx

        elif tx_type == "ORDER_FILL":
            trade_id = tx.get("tradeOpened", {}).get("tradeID")
            if not trade_id:
                continue  # Skip fills without opened trade

            existing = table.first(formula=f"{{Fill ID}} = '{tx['id']}'")

            if existing:
                # Check if it's a closing fill
                if "tradesClosed" in tx:
                    logger.info(f"🔁 Updating Airtable Record with CLOSE data:\n{json.dumps(tx, indent=2)}")
                    fields = {
                        "Exit Price": float(tx["price"]),
                        "Realized P/L ($)": float(tx["pl"]),
                        "Account Balance After": float(tx["accountBalance"]),
                        "Reason": tx.get("reason"),
                        "Trade State": "CLOSED"
                    }
                    table.update(existing["id"], fields)
                continue

            fields = {
                "Fill ID": tx["id"],
                "OANDA Order ID": tx.get("orderID"),
                "Instrument": tx.get("instrument"),
                "Order Type": tx.get("reason", "UNKNOWN"),
                "Direction": "Long" if int(tx["units"]) > 0 else "Short",
                "Units": abs(int(tx["units"])),
                "Entry Price": float(tx["price"]),
                "Filled Price": float(tx["price"]),
                "Execution Time": tx.get("time"),
                "Order Time": tx.get("time"),
                "Order Status": "CLOSED" if "tradesClosed" in tx else "FILLED",
                "Realized P/L ($)": float(tx.get("pl", 0.0)),
                "Account Balance After": float(tx.get("accountBalance", 0.0)),
                "Spread Cost": float(tx.get("halfSpreadCost", 0.0)),
                "Initial Margin Required": float(tx.get("tradeOpened", {}).get("initialMarginRequired", 0.0)),
                "Financing ($)": float(tx.get("financing", 0.0)),
                "Margin Used ($)": float(tx.get("tradeOpened", {}).get("initialMarginRequired", 0.0)),
                "Reason": tx.get("reason"),
                "Trade State": "CLOSED" if "tradesClosed" in tx else "FILLED",
            }

            if trade_id in stop_loss_by_trade_id:
                fields["Stop Loss"] = float(stop_loss_by_trade_id[trade_id].get("price", 0.0))
            if trade_id in take_profit_by_trade_id:
                fields["Target Price"] = float(take_profit_by_trade_id[trade_id].get("price", 0.0))

            logger.info(f"📤 Creating Airtable Record for FILL:\n{json.dumps(fields, indent=2)}")
            table.insert(fields)

    save_last_transaction_id(latest_tx_id)

if __name__ == "__main__":
    sync_new_fills()


In [ ]:
import os
import sys
import json
import logging
from oandapyV20 import API
from oandapyV20.endpoints.transactions import TransactionIDRange
from oandapyV20.endpoints.trades import OpenTrades
from datetime import datetime

# Add root directory to path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from pyairtable import Api
from config.oanda_config import API_KEY, ACCOUNT_ID
from config.airtable_config import AIRTABLE_API_TOKEN, BASE_ID, TABLE_NAME

# Airtable setup
airtable_api = Api(AIRTABLE_API_TOKEN)
table = airtable_api.table(BASE_ID, TABLE_NAME)

# Logging setup
LOG_FILE = os.path.join(os.path.dirname(__file__), "../logs/sync.log")
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode="a",
)
logger = logging.getLogger(__name__)

# Load/save sync ID helpers
def load_last_transaction_id():
    try:
        with open("last_sync.json", "r") as f:
            return json.load(f).get("last_transaction_id", "0")
    except:
        return "0"

def save_last_transaction_id(tx_id):
    try:
        with open("last_sync.json", "w") as f:
            json.dump({"last_transaction_id": tx_id}, f)
        logger.info(f"💾 Saved last transaction ID: {tx_id}")
    except Exception as e:
        logger.error(f"❌ Failed to save last transaction ID: {e}")

# --- 1. Sync New Fills ---
def sync_new_fills():
    logger.info("📄 Syncing new fills...")
    client = API(access_token=API_KEY, environment="practice")
    last_id = load_last_transaction_id()

    r = TransactionIDRange(accountID=ACCOUNT_ID, params={"from": str(int(last_id) + 1), "to": "99999999"})
    client.request(r)

    transactions = r.response.get("transactions", [])
    for tx in transactions:
        print("📩 Transaction received:\n", json.dumps(tx, indent=2))
        tx_type = tx.get("type")

        if tx_type == "ORDER_FILL":
            # OPEN TRADE
            if "tradeOpened" in tx:
                trade_id = tx["tradeOpened"]["tradeID"]
                existing = table.first(formula=f"{{Fill ID}} = '{trade_id}'")

                fields = {
                    "Fill ID": trade_id,
                    "OANDA Order ID": tx["orderID"],
                    "Instrument": tx["instrument"],
                    "Order Type": tx.get("reason", "ORDER_FILL"),
                    "Direction": "Long" if int(tx["units"]) > 0 else "Short",
                    "Units": abs(int(tx["units"])),
                    "Entry Price": float(tx["price"]),
                    "Filled Price": float(tx["price"]),
                    "Execution Time": tx.get("time"),
                    "Order Time": tx.get("time"),
                    "Order Status": "FILLED",
                    "Realized P/L ($)": float(tx.get("pl", 0.0)),
                    "Account Balance After": float(tx.get("accountBalance", 0.0)),
                    "Spread Cost": float(tx.get("halfSpreadCost", 0.0)),
                    "Reason": tx.get("reason", "ORDER_FILL"),
                    "Initial Margin Required": float(tx["tradeOpened"].get("initialMarginRequired", 0.0)),
                    "Financing ($)": float(tx.get("financing", 0.0)),
                    "Margin Used ($)": float(tx["tradeOpened"].get("initialMarginRequired", 0.0)),
                    "Trade State": "FILLED",
                }

                # Add stop loss / take profit if present in LIMIT_ORDER
                if "stopLossOnFill" in tx:
                    fields["Stop Loss"] = float(tx["stopLossOnFill"].get("price", 0.0))
                if "takeProfitOnFill" in tx:
                    fields["Target Price"] = float(tx["takeProfitOnFill"].get("price", 0.0))

                print("📤 Creating Airtable Record for FILL:\n", json.dumps(fields, indent=2))
                if not existing:
                    table.create(fields)
                else:
                    table.update(existing["id"], fields)

            # CLOSE TRADE
            elif "tradesClosed" in tx and tx["tradesClosed"]:
                trade_id = tx["tradesClosed"][0]["tradeID"]
                existing = table.first(formula=f"{{Fill ID}} = '{trade_id}'")

                if not existing:
                    logger.warning(f"⚠️ No matching open trade found for closed trade {trade_id}")
                    continue

                fields = {
                    "Exit Price": float(tx["price"]),
                    "Realized P/L ($)": float(tx.get("pl", 0.0)),
                    "Account Balance After": float(tx.get("accountBalance", 0.0)),
                    "Reason": tx.get("reason", "CLOSE"),
                    "Execution Time": tx.get("time"),
                    "Order Status": "CLOSED",
                    "Trade State": "CLOSED",
                    "Financing ($)": float(tx.get("financing", 0.0)),
                    "Spread Cost": float(tx.get("halfSpreadCost", 0.0))
                }

                print("🔁 Updating Airtable Record with CLOSE data:\n", json.dumps(fields, indent=2))
                table.update(existing["id"], fields)

    latest_id = r.response.get("lastTransactionID", last_id)
    save_last_transaction_id(latest_id)

# --- 2. Sync Open Trades (Unrealized P&L) ---
def sync_open_trades():
    logger.info("🔄 Fetching open trades for Airtable sync...")
    client = API(access_token=API_KEY, environment="practice")
    r = OpenTrades(accountID=ACCOUNT_ID)
    client.request(r)
    open_trades = r.response.get("trades", [])
    logger.info(f"✅ {len(open_trades)} open trades retrieved.")

    for trade in open_trades:
        trade_id = trade["id"]
        existing = table.first(formula=f"{{Fill ID}} = '{trade_id}'")

        if not existing:
            logger.warning(f"⚠️ No Airtable record for open trade {trade_id}")
            continue

        fields = {
            "Trade State": trade.get("state", "OPEN"),
            "Unrealized P/L ($)": float(trade.get("unrealizedPL", 0.0)),
            "Initial Margin Required": float(trade.get("initialMarginRequired", 0.0)),
            "Margin Used": float(trade.get("marginUsed", 0.0)),
            "Financing ($)": float(trade.get("financing", 0.0)),
        }

        if "takeProfitOrder" in trade:
            fields["Target Price"] = float(trade["takeProfitOrder"].get("price", 0.0))
        if "stopLossOrder" in trade:
            fields["Stop Loss"] = float(trade["stopLossOrder"].get("price", 0.0))

        print("🔄 Updating Airtable Record for Open Trade:\n", json.dumps(fields, indent=2))
        table.update(existing["id"], fields)

# --- Main ---
if __name__ == "__main__":
    sync_new_fills()
    sync_open_trades()
